In [ ]:
import numpy as np
from glob import glob
import pandas as pd
import random
from tqdm import tqdm
from sklearn import metrics
from sklearn.metrics import auc
import pickle
import pyod
from pyod.utils.stat_models import pairwise_distances_no_broadcast
import warnings
from icecream import ic
from src.constant import translation_dict
from torch.utils.data import random_split, DataLoader
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
import time
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn import metrics
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold

%load_ext autoreload
%autoreload 2
warnings.filterwarnings("ignore")

In [ ]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:15000")
mlflow.set_experiment("my-experiment")

In [ ]:
import pandas as pd
import glob

# ?????CSV??
csv_path = 'data/Train'
pkl_path = 'data/pkl/Train'
csv_files = glob.glob(csv_path+'/*.csv')
pkl_files = glob.glob(pkl_path+'/*.pkl')

In [ ]:
df_list = [pd.read_csv(f).replace({'label': {10: 1}}).assign(sn=i) for i, f in enumerate(csv_files)]
df_list = pd.concat(df_list)
df_list.shape


In [ ]:
from src.statistical_analyzer import StatisticalAnalyzer


StatisticalAnalyzer.show_report(df_list, 'data/report/report.html')

# ????

In [ ]:
display_df = df_list[df_list["sn"]==df_list["sn"].unique()[1]]
# display_df.head().to_csv('data/output.csv', index=False)

In [ ]:
display_df.to_csv('data/display.csv', index=False)

In [ ]:
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go

# ???????
custom_colors = [
    "#7ec2f3",
    "#76d0a3",
    "#f79992",
    "#fa90ba",
    "#faaf76",
    "#b3acf2",
    "#ffdd76",
    "#76cdd3",
    "#e9e9e9",
]

def plot_lines(df,x_col,width=800,height=600 , **y_cols):
    fig = sp.make_subplots(specs=[[{"secondary_y": True}]])

    for i, (y_col, y_col_name) in enumerate(y_cols.items()):
        color = custom_colors[i % len(custom_colors)]
        trace = go.Scatter(x=x_col, y=df[y_col], mode='lines', name=y_col_name, line=dict(color=color))
        secondary_y = i > 0
        
        fig.add_trace(trace, secondary_y=secondary_y)
        if secondary_y:
            # ???y?
            axis_name = f"yaxis{i + 1}"
            fig.update_layout(
                **{axis_name: dict(
                title=y_col_name,
                titlefont=dict(color=color),
                tickfont=dict(color=color),
                anchor="x",
                overlaying="y",
                side="right",
                position=1 - i * 0.05,
                showgrid=False
            )})
        else:
            # ????y?
            fig.update_yaxes(title_text=y_col_name, showgrid=False)

    # ??x?
    # fig.update_xaxes(title_text=str(x_col), showgrid=False)

    # ???????
    fig.update_layout(
        title_text='?'.join(y_cols.values()),
        autosize=False,
        width=width,
        height=height,
        title_x=0.1,
        # plot_bgcolor='white',
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(showgrid=False),
        font=dict(
            family="Courier New, monospace",
            size=30,
            color="black"
        )
    )
    fig.update_yaxes(showline=True, linewidth=2, linecolor='rgba(0,0,0,0)', mirror=True)

    # fig.show()
    return fig

plot_lines(df=display_df, x_col=display_df.index,width=1600,height=600, soc='??', volt='??')


In [ ]:
# df_list.to_csv('all_battery.csv', index=False)

In [ ]:
numeric_col=df_list.select_dtypes(include=[np.number]).columns
df_list[numeric_col]=df_list[numeric_col].astype('float32')
df_list['label'] = df_list['label'].astype('int8')
df_list['sn'] = df_list['sn'].astype('int32')
ic(numeric_col)
df_list.info()
df_list.head()

In [ ]:
# df_list.groupby('label').apply(lambda x: x.sort_values(['sn', 'timestamp']))

df_last = df_list.groupby('label').apply(lambda x: x.sort_values(['sn', 'timestamp'])).groupby('sn').last()
df_last.shape

In [ ]:
# df_last.to_csv('df_last.csv', index=False)

In [ ]:
# random.shuffle(df_list)

# ?????
df = df_last
df = df.reset_index(drop=True)

# df.head()


In [ ]:

# ??????????
X_list = df.drop(['label','timestamp'], axis=1)
y_list = df['label']
ic(X_list.shape)
X_list.head()


In [ ]:
n_splits = 5
kf = KFold(n_splits=n_splits)  # 5 ?????

accuracies = []

for train_index, test_index in kf.split(X=X_list, y=y_list):
    ic("TRAIN:", train_index, "TEST:", test_index)

    X_train, X_test = X_list[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    clf = CatBoostClassifier(verbose=0, n_estimators=100)
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    accuracies.append(accuracy)

# ???????
average_accuracy = np.mean(accuracies)
print(f'Average accuracy: {average_accuracy}')

# ?????

In [ ]:
test_size=0.3
# ????
X_train, X_test, y_train, y_test = train_test_split(X_list, y_list, test_size=test_size, random_state=2024)


# ic(X_train.shape)
# ic(X_test.shape)
# ic(y_train.shape)
# ic(y_test.shape);

In [ ]:
# K ?????

In [ ]:

# ??? KFold ?
from sklearn.model_selection import KFold
n_splits = 5

kf = KFold(n_splits=n_splits)  # 5 ?????

accuracies = []

for train_index, test_index in kf.split(X_list):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    clf = CatBoostClassifier(verbose=0, n_estimators=100)
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    accuracies.append(accuracy)

# ???????
average_accuracy = np.mean(accuracies)
print(f'Average accuracy: {average_accuracy}')

In [ ]:

import xgboost as xgb
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.01,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0,
    'reg_alpha': 0,
    'reg_lambda': 1,
    'seed': 7,
    'nthread': -1,
    'use_label_encoder': False
}

clf = xgb.XGBClassifier(
            **xgb_params
        )

clf.fit(X_train, y_train)



In [ ]:
# import lightgbm as lgb
# params = {'num_leaves': 60, #???????????,?????,????????
#           'min_data_in_leaf': 30,
#           'objective': 'binary', #???????
#           'max_depth': 5,
#           'learning_rate': 0.03,
#           "min_sum_hessian_in_leaf": 6,
#           "boosting": "gbdt",
#           "feature_fraction": 0.9,  #???????
#           "bagging_freq": 1,
#           "bagging_fraction": 0.8,
#           "bagging_seed": 11,
#           "lambda_l1": 0.1,             #l1??
#           'lambda_l2': 0.001,     #l2??
#           "verbosity": -1,
#           "nthread": -1,                #????,-1??????,????,???????
#           'metric': {'binary_logloss', 'auc'},  ##??????
#           "random_state": 2019, #?????,??????????????
#           # 'device': 'gpu' ##??????gpu???lightgbm,??????
#           }
# clf = lgb.LGBMClassifier()
# clf.fit(X_train, y_train)

In [ ]:
# from catboost import CatBoostClassifier
# clf = CatBoostClassifier(verbose=0, n_estimators=100)
# clf.fit(X_train, y_train)


In [ ]:
y_pred = clf.predict_proba(X_test)

def adjust_threshold(probs, threshold):
    # ??????????(???????,??????)
    positive_probs = probs[:, 1]

    # ????????????
    y_pred = [1 if prob >= threshold else 0 for prob in positive_probs]

    return y_pred
    
y_pred = adjust_threshold(y_pred, 0.5)
# sum(y_pred)


In [ ]:
from src.model_evaluator import ModelEvaluator


res_eva = ModelEvaluator.evaluate(y_test, y_pred)
matrix = confusion_matrix(y_test, y_pred)

ModelEvaluator.draw_confusion_matrix(c_matrix=matrix, classes = ['??', '??'])


# mlflow log

In [ ]:
print(os.environ["AWS_ACCESS_KEY_ID"])
print(os.environ["AWS_SECRET_ACCESS_KEY"])

In [ ]:
import boto3
from botocore.client import Config

# MinIO ?????
minio_endpoint = 'http://192.168.31.25:9000'
minio_access_key = 'minio'
minio_secret_key = 'minio123'
bucket_name = 'mlflow'
file_path = './ml.ipynb'
object_name = 'file.txt'  # ??? MinIO ?????

# ?? S3 ???
s3_client = boto3.client(
    's3',
    endpoint_url=minio_endpoint,
    aws_access_key_id=minio_access_key,
    aws_secret_access_key=minio_secret_key,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# ?? bucket(?????)
try:
    s3_client.head_bucket(Bucket=bucket_name)
except:
    s3_client.create_bucket(Bucket=bucket_name)

# ????
s3_client.upload_file(file_path, bucket_name, object_name)

print(f"File {file_path} uploaded to {bucket_name}/{object_name}")


In [ ]:
import requests
response = requests.get("http://192.168.31.25:9001")
print(response.status_code)

In [ ]:
print(mlflow.get_artifact_uri())



In [ ]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")
# mlflow.set_tracking_uri(uri="./mlflow_test/")
# os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'http://127.0.0.1:9000'
# os.environ['AWS_ACCESS_KEY_ID'] = 'minio'
# os.environ['AWS_SECRET_ACCESS_KEY'] = 'minio123'
from mlflow.models import infer_signature
# Create a new MLflow Experiment
mlflow.set_experiment("MLflow Qqqckstart")
import os


mlflow.end_run()

# Start an MLflow run
with mlflow.start_run() as run:
    ic(run.info.run_id)

    # Log the hyperparameters
    mlflow.log_params(xgb_params)
    # Log the loss metric
    mlflow.log_metrics(res_eva)
    # mlflow.log_metric("precision", res_eva["precision"])
    # mlflow.log_metric("recall", res_eva["recall"])
    # mlflow.log_metric("specificity", res_eva["specificity"])
    import psutil

    # mlflow.log_metric("accuracy", res_eva["accuracy"])
    # cpu_percent = psutil.cpu_percent()
    # virtual_memory = psutil.virtual_memory().percent
    mlflow.system_metrics.enable_system_metrics_logging()
    time.sleep(5)
    # mlflow.log_("cpu_percent", cpu_percent)
    # mlflow.log_metric("virtual_memory_percent", virtual_memory) 
    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("????", "????")
    mlflow.set_tag("??", "???")
    # Infer the model signature
    signature = infer_signature(X_train, clf.predict(X_train))
    # os.environ['AWS_ACCESS_KEY_ID'] = 'Z0qLwDccsRYWpKNadFYo'
    # os.environ['AWS_SECRET_ACCESS_KEY'] = '26QXPzDoLDm7kVsszwizjNN1AMjudl5Il7oNQuw6'
    # os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'http://192.168.31.25:9000'
    # os.environ['AWS_MAX_ATTEMPTS'] = '1'
    mlflow.log_artifact('./ml.ipynb')
    model_info = mlflow.sklearn.log_model(
        sk_model=clf,
        artifact_path="mlflow/mlflow",
        signature=signature,
        input_example=X_train,
        registered_model_name="tracking-quickstart",
        
    )

In [ ]:
# ????
import mlflow.pyfunc

# Specify the run_id and artifact path
run_id = "7f055ccb9de44ad3beac98861c5015a9"
artifact_path = "mlflow/mlflow"

# Load the model
model_uri = f"runs:/{run_id}/{artifact_path}"
model = mlflow.pyfunc.load_model(model_uri)
# Now you can use the model object to make predictions
# For example:
# Assuming X_test is a pandas DataFrame
X_test.columns = ['f{}'.format(i) for i in range(len(X_test.columns))]


# Now you can use the model object to make predictions
predictions = model.predict(X_test.iloc[[20]])
predictions
X_test.iloc[20].to_frame().T


In [ ]:
# ????? flask ??
model.serve(port=5000)


In [ ]:
import mlflow.data
import pandas as pd
from mlflow.data.pandas_dataset import PandasDataset


dataset_source_url = "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv"
raw_data = pd.read_csv(dataset_source_url, delimiter=";")
raw_data

In [ ]:

# Create an instance of a PandasDataset
dataset = mlflow.data.from_pandas(
    raw_data, source=dataset_source_url, name="wine quality - white", targets="quality"
)

# ??onnx ??

In [ ]:
import onnxmltools
from onnxmltools.convert import convert_xgboost, convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]

# Convert XGBoost model to ONNX
onnx_model = convert_xgboost(clf, initial_types=initial_type)

# Log model in ONNX format
with mlflow.start_run():
    mlflow.onnx.log_model(onnx_model, "model")

# ??Onnx

In [ ]:
import mlflow.onnx

# Specify the run_id and artifact path
run_id = run.info.run_id
artifact_path = "model"

# Load the ONNX model
model_uri = f"runs:/{run_id}/{artifact_path}"
onnx_model = mlflow.onnx.load_model(model_uri)

In [ ]:
# Assuming X_train is your training data
# ????? xgb ????? ????
new_feature_names = ['f{}'.format(i) for i in range(X_train.shape[1])]
X_train.columns = new_feature_names

# Train your XGBoost model
clf.fit(X_train, y_train)

# Now you can convert your model to ONNX format
initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
onnx_model = convert_xgboost(clf, initial_types=initial_type)

# ????MLflow??
with mlflow.start_run():
    # ?????ONNX??
    mlflow.onnx.log_model(onnx_model, "model", registered_model_name="my_onnx_model")

In [ ]:
# ????
import mlflow

# ????????MLflow??????????
run_id = run.info.run_id
model_uri = "runs:/" + run_id + "/model"

# ????
mlflow.register_model(
    model_uri,
    "my_model"
)

In [ ]:
# onnx????
import mlflow.onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Convert sklearn XGBoost model to ONNX
initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
onnx_model = convert_sklearn(clf, initial_types=initial_type)

# Log model in ONNX format
with mlflow.start_run():
    mlflow.onnx.log_model(onnx_model, "model")

# Load model in ONNX format
loaded_model = mlflow.onnx.load_model("runs:/<run_id>/model")

# ????

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
ic(model_info)
ic(model_info.model_uri)

In [ ]:
predictions = loaded_model.predict(X_test)

iris_feature_names = X_train.columns

# Convert X_test validation feature data to a Pandas DataFrame
result = pd.DataFrame(X_test, columns=iris_feature_names)

# Add the actual classes to the DataFrame
result["actual_class"] = y_test

# Add the model predictions to the DataFrame
result["predicted_class"] = predictions


In [ ]:
version: "3.3" services: db: restart: always image: postgres container_name: mlflow_db expose: - "${PG_PORT}" networks: - backend environment: - POSTGRES_USER=${PG_USER} - POSTGRES_PASSWORD=${PG_PASSWORD} - POSTGRES_DATABASE=${PG_DATABASE} volumes: - ./db_data:/var/lib/postgresql/data/ healthcheck: test: ["CMD", "pg_isready", "-p", "${PG_PORT}", "-U", "${PG_USER}"] interval: 5s timeout: 5s retries: 3 s3: restart: always image: minio/minio container_name: mlflow_minio volumes: - ./minio_data:/data ports: - "${MINIO_PORT}:9000" - "${MINIO_CONSOLE_PORT}:9001" networks: - frontend - backend environment: - MINIO_ROOT_USER=${MINIO_ROOT_USER} - MINIO_ROOT_PASSWORD=${MINIO_ROOT_PASSWORD} - MINIO_ADDRESS=${MINIO_ADDRESS} - MINIO_PORT=${MINIO_PORT} - MINIO_STORAGE_USE_HTTPS=${MINIO_STORAGE_USE_HTTPS} - MINIO_CONSOLE_ADDRESS=${MINIO_CONSOLE_ADDRESS} command: server /data healthcheck: test: ["CMD", "curl", "-f", "http://localhost:9000/minio/health/live"] interval: 30s timeout: 20s retries: 3 tracking_server: restart: always build: ./mlflow image: mlflow_server container_name: mlflow_server depends_on: - db ports: - "${MLFLOW_PORT}:5000" networks: - frontend - backend environment: - AWS_ACCESS_KEY_ID=${MINIO_ACCESS_KEY} - AWS_SECRET_ACCESS_KEY=${MINIO_SECRET_ACCESS_KEY} - MLFLOW_S3_ENDPOINT_URL=http://s3:${MINIO_PORT} - MLFLOW_S3_IGNORE_TLS=true command: > mlflow server --backend-store-uri postgresql://${PG_USER}:${PG_PASSWORD}@db:${PG_PORT}/${PG_DATABASE} --host 0.0.0.0 --serve-artifacts --artifacts-destination s3://${MLFLOW_BUCKET_NAME} healthcheck: test: ["CMD", "curl", "-f", "http://localhost:${MLFLOW_PORT}/"] interval: 30s timeout: 10s retries: 3 volumes: db_data: minio_data: networks: frontend: driver: bridge backend: driver: bridge
